# LeetCode 2332 — The Latest Time to Catch a Bus bằng Python

Link đề: https://leetcode.com/problems/the-latest-time-to-catch-a-bus/ · Độ khó: **Medium** · Chủ đề: Array, Two Pointers, Binary Search, Sorting

Skeleton LeetCode:

```python
class Solution:
    def latestTimeCatchTheBus(self, buses: List[int], passengers: List[int], capacity: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Mô phỏng đúng quy tắc xếp hàng: sắp xếp xe và hành khách, mỗi xe chở tối đa `capacity` người **đến sớm nhất** còn đang chờ (đến lúc `≤` giờ xe chạy).
2. Nhận ra chỉ **xe cuối cùng** quyết định đáp án: hoặc xe cuối còn chỗ (đến đúng giờ xe chạy là tốt nhất), hoặc xe cuối đầy (phải đến trước người cuối cùng lên xe).
3. Xử lý ràng buộc "không được đến **cùng lúc** với hành khách khác" bằng cách **lùi dần** qua các thời điểm đã có người.
4. Một cách nhìn khác: khả năng lên xe **đơn điệu** theo thời gian đến ⇒ **tìm nhị phân** thời điểm muộn nhất, rồi lùi qua các thời điểm trùng.
5. Kiểm chứng bằng vét cạn: thử mọi thời điểm đến, chèn "mình" vào hàng và mô phỏng.

## 2. Đề bài & ví dụ

Cho `buses` (giờ xuất phát của các xe, phân biệt), `passengers` (giờ đến của hành khách, phân biệt), và `capacity`. Hành khách đến thì xếp hàng chờ xe kế tiếp. Lên được xe chạy lúc `x` nếu đến lúc `y ≤ x` và xe chưa đầy; người đến sớm hơn lên trước. Trả về thời điểm **muộn nhất** mình có thể đến bến mà vẫn lên được một xe. Mình **không được** đến cùng thời điểm với hành khách khác. Hai mảng không nhất thiết đã sắp xếp.

**Ví dụ 1:** `buses = [10,20]`, `passengers = [2,17,18,19]`, `capacity = 2` → `16`.

**Ví dụ 2:** `buses = [20,30,10]`, `passengers = [19,13,26,4,25,11,21]`, `capacity = 2` → `20`.

**Ràng buộc:** `1 ≤ n, m, capacity ≤ 10⁵`; `2 ≤ buses[i], passengers[i] ≤ 10⁹`.

In [1]:
from typing import List
from bisect import bisect_left
import random, time

examples = [([10, 20], [2, 17, 18, 19], 2, 16), ([20, 30, 10], [19, 13, 26, 4, 25, 11, 21], 2, 20)]
for b, p, c, exp in examples:
    print(f"buses={sorted(b)} passengers={sorted(p)} capacity={c} -> kỳ vọng {exp}")

buses=[10, 20] passengers=[2, 17, 18, 19] capacity=2 -> kỳ vọng 16
buses=[10, 20, 30] passengers=[4, 11, 13, 19, 21, 25, 26] capacity=2 -> kỳ vọng 20


## 3. Kiến thức nền

### 3.1 Mô phỏng hàng chờ với hai con trỏ

Sắp xếp `buses` và `passengers`. Con trỏ `j` chỉ hành khách đầu hàng. Với mỗi xe `t` (theo thứ tự): lên xe khi còn chỗ và `passengers[j] ≤ t`.

In [2]:
def simulate(buses, passengers, capacity):
    B, P = sorted(buses), sorted(passengers); j = 0; log = []
    for t in B:
        c = capacity; on = []
        while c and j < len(P) and P[j] <= t:
            on.append(P[j]); c -= 1; j += 1
        log.append((t, on, c))
    return log

for t, on, c in simulate(*examples[1][:3]):
    print(f"xe {t}: chở {on}, còn trống {c}")

xe 10: chở [4], còn trống 1
xe 20: chở [11, 13], còn trống 0
xe 30: chở [19, 21], còn trống 0


### 3.2 Đơn điệu: đến sớm hơn không bao giờ làm mất chỗ

Nếu đến lúc `T` mà lên được xe, thì đến lúc `T' < T` (không trùng ai) cũng lên được: mình đứng **trước** (hoặc cùng vị trí) trong hàng so với khi đến lúc `T`, và có thể bắt các xe sớm hơn. Tính chất này cho phép tìm nhị phân.

### 3.3 Lùi qua các thời điểm đã bị chiếm

Nếu ứng viên `T` trùng giờ đến của một hành khách, thử `T − 1`, `T − 2`, … cho tới khi gặp một thời điểm trống. Vì giờ đến phân biệt và đã sắp xếp, lùi theo con trỏ `j` là O(số người trùng liên tiếp).

## 4. Ý tưởng nền tảng / trực giác

Chỉ cần xét **xe cuối cùng** (mình muốn đến càng muộn càng tốt; xe cuối chạy muộn nhất).

- Nếu xe cuối **còn chỗ** sau khi mọi người hợp lệ đã lên: mình đến đúng lúc xe cuối chạy (`buses[-1]`) là lên được — trừ khi có ai đến đúng lúc đó, thì lùi lại.
- Nếu xe cuối **đầy**: mình phải chiếm chỗ của người **cuối cùng** lên xe cuối ⇒ đến trước họ: `P[last] − 1`, và lùi tiếp nếu trùng.

Trong cả hai trường hợp, các thời điểm bị chặn khi lùi là **giờ của các hành khách đã lên xe** (đứng liền nhau ở cuối danh sách lên xe) — vì vậy lùi bằng con trỏ `j` đi ngược.

Tính tay ví dụ 1: sắp `B = [10, 20]`, `P = [2, 17, 18, 19]`, `capacity = 2`. Xe 10 chở `2`; xe 20 chở `17, 18` (đầy; 19 phải chờ — không có xe sau). Xe cuối đầy ⇒ ứng viên `18 − 1 = 17` — trùng hành khách `17` ⇒ `16` ✔.

Ví dụ 2: `B = [10, 20, 30]`, `P = [4, 11, 13, 19, 21, 25, 26]`. Xe 10: `4`; xe 20: `11, 13`; xe 30: `19, 21` (đầy). Ứng viên `21 − 1 = 20` — không trùng ⇒ `20` ✔.

In [3]:
for b, p, c, exp in examples:
    log = simulate(b, p, c); t_last, on_last, left = log[-1]
    print(f"xe cuối {t_last}: chở {on_last}, còn trống {left} ->",
          "bắt đầu từ giờ xe chạy" if left else f"bắt đầu từ {on_last[-1]} - 1", "| kỳ vọng", exp)

xe cuối 20: chở [17, 18], còn trống 0 -> bắt đầu từ 18 - 1 | kỳ vọng 16
xe cuối 30: chở [19, 21], còn trống 0 -> bắt đầu từ 21 - 1 | kỳ vọng 20


## 5. Cách 1 — Brute Force / baseline

### 5.1 Thử mọi thời điểm đến từ muộn tới sớm, chèn mình vào hàng và mô phỏng — O(T · (n + m) log)

In [4]:
def can_catch(buses, passengers, capacity, T):
    if T in set(passengers): return False
    B = sorted(buses); P = sorted(passengers + [T]); j = 0
    for t in B:
        c = capacity
        while c and j < len(P) and P[j] <= t:
            if P[j] == T: return True
            c -= 1; j += 1
    return False

def brute(buses, passengers, capacity):
    for T in range(max(buses), 0, -1):
        if can_catch(buses, passengers, capacity, T): return T
    return -1

for b, p, c, exp in examples:
    print(brute(b, p, c), "OK" if brute(b, p, c) == exp else "SAI")

16 OK
20 OK


Đến sau `max(buses)` chắc chắn trễ mọi xe, nên chỉ cần thử `T ≤ max(buses)`. Nhưng `max(buses)` tới `10⁹` ⇒ quá chậm; chỉ dùng kiểm chứng với giá trị nhỏ.

### 5.2 Baseline 2 — Chỉ thử các **ứng viên**: `buses[-1]` và `p − 1` với mọi hành khách `p`

Đáp án tối ưu luôn là "giờ xe cuối" hoặc "ngay trước một hành khách nào đó" (vì giữa hai mốc liên tiếp, đến càng muộn càng tốt). Có `m + 1` ứng viên, mỗi ứng viên mô phỏng O(n + m) ⇒ O(m · (n + m)).

In [5]:
def candidates(buses, passengers, capacity):
    pset = set(passengers)
    cands = sorted({max(buses)} | {p - 1 for p in passengers if p - 1 >= 1}, reverse=True)
    for T in cands:
        if T not in pset and can_catch(buses, passengers, capacity, T): return T
    return -1

for b, p, c, exp in examples:
    print(candidates(b, p, c), "OK" if candidates(b, p, c) == exp else "SAI")

16 OK
20 OK


**Nút thắt:** mỗi ứng viên mô phỏng lại từ đầu. Nhưng một lần mô phỏng **không có mình** đã cho đủ thông tin: trạng thái của xe cuối và danh sách người lên xe ⇒ O(1) ứng viên thật sự cần xét, rồi lùi.

## 6. Ý tưởng thuật toán / bước đột phá

```
sort(buses); sort(passengers)
j = 0
for t in buses:
    c = capacity
    while c > 0 and j < m and passengers[j] <= t: c -= 1; j += 1
# j = số người đã lên xe; c = số chỗ trống của XE CUỐI
j -= 1                                   # chỉ số người cuối cùng đã lên xe (−1 nếu không ai)
ans = buses[-1] if c > 0 else passengers[j]
while j >= 0 and passengers[j] == ans:   # lùi qua các thời điểm đã có người
    ans -= 1; j -= 1
return ans
```

- Khi `c > 0`: bắt đầu từ `buses[-1]`; nếu hành khách cuối lên xe đến đúng lúc đó thì lùi.
- Khi `c = 0`: bắt đầu từ `passengers[j]` (người cuối lên xe) rồi vòng lùi đầu tiên chắc chắn chạy (vì `passengers[j] == ans`) ⇒ ra `passengers[j] − 1`, tiếp tục lùi nếu trùng.

**Độ phức tạp:** O(n log n + m log m) cho sắp xếp, O(n + m) cho mô phỏng và lùi.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mô phỏng + lùi (cách doocs)

In [6]:
def simulate_back(buses, passengers, capacity):
    B, P = sorted(buses), sorted(passengers); j = 0
    for t in B:
        c = capacity
        while c and j < len(P) and P[j] <= t:
            c, j = c - 1, j + 1
    j -= 1
    ans = B[-1] if c else P[j]
    while ~j and P[j] == ans:
        ans, j = ans - 1, j - 1
    return ans

for b, p, c, exp in examples:
    print(simulate_back(b, p, c), "OK" if simulate_back(b, p, c) == exp else "SAI")

16 OK
20 OK


`~j` là `-(j + 1)`, khác 0 ⇔ `j ≠ −1` ⇔ còn hành khách để xét — mẹo gọn cho `j >= 0`.

### 7.2 Cách 3 — Tìm nhị phân thời điểm muộn nhất (bỏ qua ràng buộc trùng), rồi lùi qua thời điểm trùng

Định nghĩa `ok(T)` = "nếu được phép đến lúc `T` **và xếp trước** mọi người đến cùng lúc `T`, mình có lên được không". `ok` đơn điệu giảm theo `T`. Tìm `T* = max{T : ok(T)}`, rồi lùi `T*` qua các giờ đến của hành khách — mọi `T ≤ T*` đều `ok`.

In [7]:
def binary_search(buses, passengers, capacity):
    B, P = sorted(buses), sorted(passengers); pset = set(P)
    def ok(T):
        # mình đứng ngay trước người đầu tiên có giờ đến >= T
        pos = bisect_left(P, T); j = 0
        for t in B:
            c = capacity
            while c and j < len(P) and P[j] <= t:
                if j == pos and T <= t: return True           # tới lượt mình (trước P[pos])
                c -= 1; j += 1
            if c and j == pos and T <= t: return True         # còn chỗ, hàng trước mình đã hết
        return False
    lo, hi = 1, B[-1]
    if not ok(lo): return -1
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if ok(mid): lo = mid
        else: hi = mid - 1
    while lo in pset: lo -= 1
    return lo

for b, p, c, exp in examples:
    print(binary_search(b, p, c), "OK" if binary_search(b, p, c) == exp else "SAI")

16 OK
20 OK


O((n + m) · log(max bus)) — chậm hơn cách 2 một hệ số log, nhưng lập luận "đơn điệu + lùi" dễ chứng minh và tổng quát hóa (ví dụ khi quy tắc lên xe phức tạp hơn).

### 7.3 Cách 4 — Mô phỏng ghi lại danh sách người lên xe cuối rồi tìm khoảng trống bằng tập hợp

In [8]:
def with_set(buses, passengers, capacity):
    B, P = sorted(buses), sorted(passengers); j = 0; last_on = []
    for t in B:
        c = capacity; last_on = []
        while c and j < len(P) and P[j] <= t:
            last_on.append(P[j]); c -= 1; j += 1
    taken = set(P)
    ans = B[-1] if c else last_on[-1] - 1
    while ans in taken: ans -= 1
    return ans

for b, p, c, exp in examples:
    print(with_set(b, p, c), "OK" if with_set(b, p, c) == exp else "SAI")

16 OK
20 OK


Lùi bằng `set` thay vì con trỏ: đơn giản hơn, cùng độ phức tạp (mỗi bước lùi O(1) trung bình, tổng số bước lùi ≤ số hành khách).

## 8. TRACE từng bước

### 8.1 Trace cách 2 trên ví dụ 1

In [9]:
def trace(buses, passengers, capacity):
    B, P = sorted(buses), sorted(passengers); j = 0
    for t in B:
        c = capacity; on = []
        while c and j < len(P) and P[j] <= t:
            on.append(P[j]); c -= 1; j += 1
        print(f"xe {t}: lên {on}, còn trống {c}")
    j -= 1
    ans = B[-1] if c else P[j]
    print("bắt đầu ans =", ans, "(giờ xe cuối)" if c else "(giờ người cuối cùng lên xe)")
    while j >= 0 and P[j] == ans:
        print(f"  {ans} trùng hành khách -> lùi"); ans -= 1; j -= 1
    print("đáp án:", ans); return ans

trace(*examples[0][:3])

xe 10: lên [2], còn trống 1
xe 20: lên [17, 18], còn trống 0
bắt đầu ans = 18 (giờ người cuối cùng lên xe)
  18 trùng hành khách -> lùi
  17 trùng hành khách -> lùi
đáp án: 16


16

### 8.2 Trace trường hợp xe cuối **còn chỗ** nhưng có người đến đúng giờ xe chạy

In [10]:
trace([10, 20], [20, 19, 3], 5)

xe 10: lên [3], còn trống 4
xe 20: lên [19, 20], còn trống 3
bắt đầu ans = 20 (giờ xe cuối)
  20 trùng hành khách -> lùi
  19 trùng hành khách -> lùi
đáp án: 18


18

Xe 20 còn chỗ, nhưng hành khách 20 và 19 đến đúng/ngay trước giờ đó ⇒ lùi xuống 18.

## 9. Code nộp LeetCode

In [11]:
from typing import List

class Solution:
    def latestTimeCatchTheBus(self, buses: List[int], passengers: List[int], capacity: int) -> int:
        buses = sorted(buses)
        passengers = sorted(passengers)
        j = 0
        for t in buses:
            seats = capacity
            while seats and j < len(passengers) and passengers[j] <= t:
                seats -= 1
                j += 1
        j -= 1                                            # người cuối cùng đã lên xe (hoặc -1)
        ans = buses[-1] if seats else passengers[j]       # xe cuối còn chỗ / đã đầy
        while j >= 0 and passengers[j] == ans:            # không được trùng giờ với ai
            ans -= 1
            j -= 1
        return ans

In [12]:
sol = Solution()
for b, p, c, exp in examples:
    got = sol.latestTimeCatchTheBus(b, p, c)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

16 | kỳ vọng 16 | OK
20 | kỳ vọng 20 | OK


Lưu ý: khi xe cuối đầy, chắc chắn `j ≥ 0` (có người lên xe cuối), nên `passengers[j]` hợp lệ. Khi xe cuối còn chỗ mà không ai lên xe nào (`j = −1`), vòng `while` không chạy và đáp án là `buses[-1]`.

## 10. Vì sao thuật toán đúng

Ký hiệu `B` đã sắp, `P` đã sắp, mô phỏng **không có mình** cho: `seats` = số chỗ trống của xe cuối, `L` = tập hành khách đã lên xe (là tiền tố `P[0..j]` của `P`, vì lên theo thứ tự đến).

**Bổ đề 1 (đơn điệu).** Nếu mình đến lúc `T` (không trùng ai) và lên được xe, thì đến lúc `T' < T` (không trùng ai) cũng lên được. *Phác thảo:* hàng chờ với mình ở `T'` khác hàng chờ với mình ở `T` chỉ ở chỗ mình đứng **sớm hơn** (trước một số người); quá trình lên xe của những người đứng trước mình không đổi, nên mình được xét lên xe không muộn hơn.

**Bổ đề 2 (điểm bắt đầu).**
- Nếu `seats > 0`: đến lúc `T = B[−1]` (nếu không trùng ai), mình đứng sau mọi người đến `< T`; những người này lên xe như trong mô phỏng gốc (mình đến sau họ nên không ảnh hưởng), xe cuối vẫn còn chỗ ⇒ mình lên. Không thể đến muộn hơn `B[−1]`.
- Nếu `seats = 0`: gọi `q = P[j]` là người cuối cùng lên xe cuối. Đến lúc `T > q` thì mọi người trong `L` đến trước mình, chiếm hết chỗ ⇒ mình không lên được. Đến lúc `T = q − 1` (nếu không trùng ai), mình đứng ngay trước `q` ⇒ chiếm chỗ của `q` ⇒ lên được.

**Bổ đề 3 (lùi).** Từ điểm bắt đầu `S` (đủ điều kiện theo Bổ đề 2 nếu không bị trùng), theo Bổ đề 1 mọi `T ≤ S` không trùng ai đều được. Đáp án là thời điểm lớn nhất `≤ S` không thuộc `P`. Các phần tử của `P` có thể bằng `S, S − 1, …` liên tiếp đều nằm trong `L` và là đuôi của `L` (vì `≤ S ≤ B[−1]` và lên xe theo thứ tự) ⇒ lùi theo con trỏ `j` đi ngược là đủ (không cần xét người ngoài `L`, vì họ đến `> S`). ∎

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi `T` | O(max(B) · (n + m) log m) | O(m) |
| 1b. Ứng viên | O(m · (n + m) log m) | O(m) |
| 2. Mô phỏng + lùi (doocs) | O(n log n + m log m) | O(n + m) |
| 3. Tìm nhị phân + lùi | O((n + m) log max(B)) | O(n + m) |
| 4. Mô phỏng + set | O(n log n + m log m) | O(m) |

In [13]:
rng = random.Random(2332)
B = rng.sample(range(2, 10**9), 10**5); P = rng.sample(range(2, 10**9), 10**5)
for cap in [1, 3, 100]:
    res = []
    for name, f in [("simulate_back", simulate_back), ("binary_search", binary_search), ("with_set", with_set),
                    ("Solution", Solution().latestTimeCatchTheBus)]:
        t0 = time.perf_counter(); res.append(f(B, P, cap)); dt = time.perf_counter() - t0
        print(f"cap={cap:<4} {name:<14} {dt*1000:8.1f} ms -> {res[-1]}")
    assert len(set(res)) == 1

cap=1    simulate_back      37.9 ms -> 997312512


cap=1    binary_search     325.6 ms -> 997312512
cap=1    with_set           59.4 ms -> 997312512
cap=1    Solution           45.1 ms -> 997312512
cap=3    simulate_back      47.9 ms -> 999982876


cap=3    binary_search     508.4 ms -> 999982876
cap=3    with_set           61.6 ms -> 999982876
cap=3    Solution           49.2 ms -> 999982876
cap=100  simulate_back      50.9 ms -> 999982876


cap=100  binary_search     537.0 ms -> 999982876
cap=100  with_set           62.5 ms -> 999982876
cap=100  Solution           50.7 ms -> 999982876


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [14]:
ALL = {"brute": brute, "candidates": candidates, "simulate_back": simulate_back, "binary_search": binary_search,
       "with_set": with_set, "Solution": Solution().latestTimeCatchTheBus}
edge = examples + [
    ([5], [10], 1, 5),                    # không ai lên được trước xe; xe còn chỗ -> đến đúng giờ xe
    ([5], [5], 1, 4),                     # người khác chiếm đúng giờ xe và chỗ duy nhất -> đến trước họ
    ([5], [5], 2, 4),                     # còn chỗ nhưng trùng giờ -> lùi
    ([10], [8, 9, 10], 3, 7),             # đầy, các giờ 8, 9, 10 liên tiếp bị chiếm
    ([3], [2], 1, 1),                     # đầy: đến trước người duy nhất
    ([10, 20], [21, 22], 1, 20),          # mọi hành khách đến sau xe cuối
]
for b, p, c, exp in edge:
    ref = brute(b, p, c)
    for name, f in ALL.items():
        assert f(list(b), list(p), c) == ref, (name, b, p, c)
    print(f"buses={b!s:<8} passengers={p!s:<10} cap={c} -> {ref} | ghi trong bảng: {exp} | khớp: {ref == exp}")

buses=[10, 20] passengers=[2, 17, 18, 19] cap=2 -> 16 | ghi trong bảng: 16 | khớp: True
buses=[20, 30, 10] passengers=[19, 13, 26, 4, 25, 11, 21] cap=2 -> 20 | ghi trong bảng: 20 | khớp: True
buses=[5]      passengers=[10]       cap=1 -> 5 | ghi trong bảng: 5 | khớp: True
buses=[5]      passengers=[5]        cap=1 -> 4 | ghi trong bảng: 4 | khớp: True
buses=[5]      passengers=[5]        cap=2 -> 4 | ghi trong bảng: 4 | khớp: True
buses=[10]     passengers=[8, 9, 10] cap=3 -> 7 | ghi trong bảng: 7 | khớp: True
buses=[3]      passengers=[2]        cap=1 -> 1 | ghi trong bảng: 1 | khớp: True
buses=[10, 20] passengers=[21, 22]   cap=1 -> 20 | ghi trong bảng: 20 | khớp: True


### 12.2 Đối chiếu ngẫu nhiên với vét cạn (seed cố định)

In [15]:
rng = random.Random(42)
for _ in range(800):
    n, m = rng.randint(1, 4), rng.randint(1, 8)
    b = rng.sample(range(2, 30), n); p = rng.sample(range(2, 30), m); cap = rng.randint(1, 3)
    ref = brute(b, p, cap)
    for name, f in ALL.items():
        assert f(list(b), list(p), cap) == ref, (name, b, p, cap)
print("800 bộ ngẫu nhiên OK")

800 bộ ngẫu nhiên OK


### 12.3 Tính chất: đáp án ≤ `max(buses)`, không trùng hành khách, và thật sự lên được xe; tăng `capacity` không làm giảm đáp án

In [16]:
rng = random.Random(7)
for _ in range(500):
    b = rng.sample(range(2, 200), rng.randint(1, 6)); p = rng.sample(range(2, 200), rng.randint(1, 20)); cap = rng.randint(1, 4)
    ans = Solution().latestTimeCatchTheBus(b, p, cap)
    assert 1 <= ans <= max(b) and ans not in p and can_catch(b, p, cap, ans)
    assert Solution().latestTimeCatchTheBus(b, p, cap + 1) >= ans
print("tính chất OK")

tính chất OK


### 12.4 Edge cực trị: một xe, `10⁵` hành khách đến liên tiếp đúng trước giờ xe, capacity rất lớn

In [17]:
N = 10**5
B = [10**9]; P = list(range(10**9 - N + 1, 10**9 + 1))          # chiếm mọi giờ từ 10^9 - N + 1 tới 10^9
print(Solution().latestTimeCatchTheBus(B, P, N + 1), "(kỳ vọng", 10**9 - N, ")")
assert Solution().latestTimeCatchTheBus(B, P, N + 1) == 10**9 - N
assert Solution().latestTimeCatchTheBus(B, P, N) == 10**9 - N       # đầy đúng N chỗ: vẫn lùi về trước người đầu

999900000 (kỳ vọng 999900000 )


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Không lùi khi xe cuối còn chỗ (trả `buses[-1]` dù có người đến đúng giờ đó)

In [18]:
def wrong_no_back_when_free(buses, passengers, capacity):
    B, P = sorted(buses), sorted(passengers); j = 0
    for t in B:
        c = capacity
        while c and j < len(P) and P[j] <= t: c -= 1; j += 1
    j -= 1
    if c: return B[-1]                                           # SAI: có thể trùng giờ một hành khách
    ans = P[j]
    while j >= 0 and P[j] == ans: ans -= 1; j -= 1
    return ans

for b, p, c in [(examples[0][0], examples[0][1], 2), ([5], [5], 2), ([10, 20], [20, 19, 3], 5)]:
    print("bản sai :", wrong_no_back_when_free(b, p, c), "| bản đúng:", Solution().latestTimeCatchTheBus(b, p, c), "cho", b, p, c)

bản sai : 16 | bản đúng: 16 cho [10, 20] [2, 17, 18, 19] 2
bản sai : 5 | bản đúng: 4 cho [5] [5] 2
bản sai : 20 | bản đúng: 18 cho [10, 20] [20, 19, 3] 5


**Vì sao sai:** ví dụ 1 (xe cuối đầy) **trùng hợp** đúng. `[5], [5], cap 2`: xe còn chỗ nhưng hành khách đến đúng lúc 5 ⇒ phải đến lúc 4.

### Lỗi 2 — Khi xe cuối đầy, trả `người cuối − 1` mà không lùi tiếp

In [19]:
def wrong_one_step(buses, passengers, capacity):
    B, P = sorted(buses), sorted(passengers); j = 0
    for t in B:
        c = capacity
        while c and j < len(P) and P[j] <= t: c -= 1; j += 1
    ans = B[-1] if c else P[j - 1] - 1                           # SAI: P[j-2] có thể bằng P[j-1] - 1
    while c and j > 0 and P[j - 1] == ans: ans -= 1; j -= 1
    return ans

for b, p, c, _ in examples:
    print("bản sai :", wrong_one_step(b, p, c), "| bản đúng:", Solution().latestTimeCatchTheBus(b, p, c))

bản sai : 17 | bản đúng: 16
bản sai : 20 | bản đúng: 20


**Vì sao sai:** ví dụ 1: người cuối lên xe là 18 ⇒ bản sai trả 17 — nhưng 17 là giờ của một hành khách. Ví dụ 2 **trùng hợp** đúng (20 không trùng ai).

### Lỗi 3 — Quên sắp xếp

In [20]:
def wrong_unsorted(buses, passengers, capacity):
    B, P = buses, passengers; j = 0                              # SAI: đề nói hai mảng không nhất thiết đã sắp
    for t in B:
        c = capacity
        while c and j < len(P) and P[j] <= t: c -= 1; j += 1
    j -= 1
    ans = B[-1] if c else P[j]
    while j >= 0 and P[j] == ans: ans -= 1; j -= 1
    return ans

for b, p, c, _ in examples:
    print("bản sai :", wrong_unsorted(b, p, c), "| bản đúng:", Solution().latestTimeCatchTheBus(b, p, c))

bản sai : 16 | bản đúng: 16
bản sai : 10 | bản đúng: 20


**Vì sao sai:** ví dụ 1 có dữ liệu đã sắp nên **trùng hợp** đúng; ví dụ 2 thì không.

### Lỗi 4 — Dùng hành khách cuối cùng **trong danh sách** thay vì người cuối cùng **đã lên xe**

In [21]:
def wrong_last_passenger(buses, passengers, capacity):
    B, P = sorted(buses), sorted(passengers); j = 0
    for t in B:
        c = capacity
        while c and j < len(P) and P[j] <= t: c -= 1; j += 1
    if c: ans = B[-1]
    else: ans = P[-1] - 1                                        # SAI: P[-1] có thể chưa lên xe
    taken = set(P)
    while ans in taken: ans -= 1
    return ans

for b, p, c, _ in examples:
    print("bản sai :", wrong_last_passenger(b, p, c), "| bản đúng:", Solution().latestTimeCatchTheBus(b, p, c))

bản sai : 16 | bản đúng: 16
bản sai : 24 | bản đúng: 20


**Vì sao sai:** ví dụ 2: người đến muộn nhất là 26 nhưng **không** lên được xe nào (xe 30 đã đầy với 19, 21). Bản sai bắt đầu từ `26 − 1 = 25` (trùng) rồi lùi tới 24 — đến lúc 24 thì 19, 21 vẫn đứng trước mình và lấp đầy xe 30 ⇒ mình không lên được. Ví dụ 1 **trùng hợp** đúng: bắt đầu từ `19 − 1 = 18`, lùi qua 18, 17 (đều có người) tới 16 — cùng kết quả với cách bắt đầu đúng từ người cuối lên xe (18). **Sửa:** bắt đầu từ người cuối cùng **đã lên xe**.

### Lỗi 5 — Kiểm tra chỗ trống của **xe đầu tiên** / tổng chỗ thay vì xe cuối

In [22]:
def wrong_total_capacity(buses, passengers, capacity):
    B, P = sorted(buses), sorted(passengers)
    eligible = [x for x in P if x <= B[-1]]
    if len(eligible) < capacity * len(B):                       # SAI: chỗ trống có thể nằm ở xe SỚM
        ans = B[-1]
    else:
        ans = eligible[capacity * len(B) - 1] - 1
    taken = set(P)
    while ans in taken: ans -= 1
    return ans

for b, p, c in [(examples[0][0], examples[0][1], 2), (examples[1][0], examples[1][1], 2), ([10, 20], [11, 12, 13], 2)]:
    print("bản sai :", wrong_total_capacity(b, p, c), "| bản đúng:", Solution().latestTimeCatchTheBus(b, p, c), "cho", b, p, c)

bản sai : 16 | bản đúng: 16 cho [10, 20] [2, 17, 18, 19] 2
bản sai : 24 | bản đúng: 20 cho [20, 30, 10] [19, 13, 26, 4, 25, 11, 21] 2
bản sai : 20 | bản đúng: 10 cho [10, 20] [11, 12, 13] 2


**Vì sao sai:** `[10, 20]`, `[11, 12, 13]`, cap 2: tổng chỗ 4 > 3 người, nhưng xe 10 chạy **trống** (không ai đến trước 10), còn xe 20 chở 11, 12 và **đầy** ⇒ 13 không lên được, và mình phải đến trước 12 ⇒ lùi qua 11 ⇒ 10. Bản sai coi "còn chỗ" ⇒ trả 20. Ví dụ 2 cũng sai: 7 người đến trước 30 so với 6 chỗ ⇒ bản sai lấy người thứ 6 (25) làm mốc, nhưng thật ra người thứ 6 **không** lên được (xe 10 chỉ có 1 người đến kịp, bỏ phí 1 chỗ). Ví dụ 1 **trùng hợp** đúng. **Sửa:** mô phỏng từng xe theo thứ tự.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Hàng đợi FIFO + nhiều "xe" có sức chứa ⇒ **sắp xếp hai mảng + hai con trỏ** mô phỏng.
- "Thời điểm muộn nhất vẫn kịp" ⇒ đơn điệu ⇒ có thể tìm nhị phân; nhưng thường chỉ cần xét **phần tử cuối** (xe cuối, người cuối).
- "Không được trùng" với tập giá trị phân biệt ⇒ **lùi** qua các giá trị liên tiếp, bằng con trỏ trên mảng đã sắp hoặc `set`.
- Viết vét cạn "chèn mình vào và mô phỏng" để đối chiếu — các bài mô phỏng có rất nhiều trường hợp biên.
- Bài cùng họ: 2071 Maximum Number of Tasks You Can Assign, 1626, 853 Car Fleet, 2141 Maximum Running Time of N Computers, 1854 Maximum Population Year.

## 15. Biến thể của bài

### 15.1 Biến thể A — Thời điểm muộn nhất để bắt được **xe thứ `k`** (theo thứ tự giờ chạy) cụ thể

In [23]:
def latest_for_bus_k(buses, passengers, capacity, k):
    B = sorted(buses)[:k + 1]                                     # chỉ quan tâm tới xe k
    P = sorted(x for x in passengers)
    # bắt được xe k (hoặc sớm hơn) => cắt bỏ các xe sau, dùng lại thuật toán
    return Solution().latestTimeCatchTheBus(B, P, capacity)

def latest_for_bus_k_brute(buses, passengers, capacity, k):
    B = sorted(buses)
    for T in range(B[k], 0, -1):
        if T in passengers: continue
        P = sorted(passengers + [T]); j = 0; got = None
        for idx, t in enumerate(B):
            c = capacity
            while c and j < len(P) and P[j] <= t:
                if P[j] == T: got = idx
                c -= 1; j += 1
        if got is not None and got <= k: return T
    return -1

rng = random.Random(151)
for _ in range(400):
    b = rng.sample(range(2, 40), rng.randint(1, 4)); p = rng.sample(range(2, 40), rng.randint(1, 8)); cap = rng.randint(1, 3)
    k = rng.randrange(len(b))
    assert latest_for_bus_k(b, p, cap, k) == latest_for_bus_k_brute(b, p, cap, k)
print([latest_for_bus_k([10, 20, 30], examples[1][1], 2, k) for k in range(3)])

[10, 12, 20]


"Bắt được một xe ≤ xe k" ⇔ bỏ các xe sau xe k rồi giải bài gốc: các xe sau không ảnh hưởng tới việc lên các xe trước.

### 15.2 Biến thể B — Số hành khách **không** lên được xe nào (không có mình)

In [24]:
def left_behind(buses, passengers, capacity):
    B, P = sorted(buses), sorted(passengers); j = 0
    for t in B:
        c = capacity
        while c and j < len(P) and P[j] <= t: c -= 1; j += 1
    return len(P) - j

print(left_behind(*examples[0][:3]), left_behind(*examples[1][:3]))

1 2


### 15.3 Biến thể C — Được phép đến **cùng lúc** với người khác (và khi đó xếp **sau** họ)

Không còn ràng buộc "không trùng", nên không cần lùi. Nếu xe cuối còn chỗ, đến đúng giờ xe cuối (xếp sau người cùng giờ nhưng vẫn còn chỗ). Nếu xe cuối đầy, đến cùng giờ với người cuối lên xe là **không** đủ (mình xếp sau họ, mất chỗ) — phải đến **trước** họ: `P[j] − 1`, và đến cùng giờ với người khác ở `P[j] − 1` (nếu có) thì xếp sau họ nhưng vẫn trước người cuối ⇒ vẫn lên được. Kiểm chứng bằng vét cạn:

In [25]:
def same_time_allowed(buses, passengers, capacity):
    B, P = sorted(buses), sorted(passengers); j = 0
    for t in B:
        c = capacity
        while c and j < len(P) and P[j] <= t: c -= 1; j += 1
    return B[-1] if c else P[j - 1] - 1

def same_time_brute(buses, passengers, capacity):
    B = sorted(buses)
    for T in range(B[-1], 0, -1):
        P = sorted([(x, 0) for x in passengers] + [(T, 1)]); j = 0   # (giờ, 1) xếp sau người cùng giờ
        for t in B:
            c = capacity; hit = False
            while c and j < len(P) and P[j][0] <= t:
                if P[j][1] == 1: hit = True
                c -= 1; j += 1
            if hit: return T
    return -1

rng = random.Random(153)
for _ in range(500):
    b = rng.sample(range(2, 30), rng.randint(1, 3)); p = rng.sample(range(2, 30), rng.randint(1, 7)); cap = rng.randint(1, 3)
    assert same_time_allowed(b, p, cap) == same_time_brute(b, p, cap), (b, p, cap)
print(same_time_allowed(*examples[0][:3]), same_time_allowed(*examples[1][:3]))

17 20


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi `T` | định nghĩa | O(max B · …) | O(m) | kiểm chứng |
| 1b. Ứng viên | `B[−1]`, `p − 1` | O(m(n+m)) | O(m) | kiểm chứng nhanh hơn |
| 2. Mô phỏng + lùi | xe cuối quyết định | O(sort) | O(n+m) | **mặc định** |
| 3. Tìm nhị phân + lùi | đơn điệu | O((n+m) log) | O(n+m) | quy tắc phức tạp hơn |
| 4. Mô phỏng + set | lùi bằng set | O(sort) | O(m) | dễ viết |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `latestTimeCatchTheBus([3], [2, 4], 2)` và `latestTimeCatchTheBus([2], [2], 1)`?

**Bài 2 (tính tay):** `buses = [5, 15]`, `passengers = [11, 12, 13, 14, 15]`, `capacity = 3`. Mô phỏng và tính đáp án.

**Bài 3 (điền TODO):** Viết `bt3_todo(buses, passengers, capacity, T)` trả về `True` ⇔ đến lúc `T` (không trùng ai) thì lên được xe.

**Bài 4 (tự viết lại):** Viết `bt4_todo(buses, passengers, capacity)` cài bản nộp không nhìn code.

**Bài 5 (mini challenge):** Viết `bt5_todo(buses, passengers, capacity)` trả về **xe** (giờ chạy) mà mình sẽ lên nếu đến vào thời điểm muộn nhất.

In [26]:
def bt3_todo(buses, passengers, capacity, T):
    # TODO: lên được xe nếu đến lúc T?
    pass

def bt4_todo(buses, passengers, capacity):
    # TODO: bản nộp
    pass

def bt5_todo(buses, passengers, capacity):
    # TODO: giờ chạy của xe mình lên
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: xe 3 chở người 2, còn 1 chỗ (người 4 đến sau xe). Mức 2: xe cuối còn chỗ ⇒ bắt đầu từ 3, không trùng. Mức 3: `3`; câu hai: xe 2 chở người 2 (đầy) ⇒ `1`.

**Bài 2.** Mức 1: xe 5: không ai. Mức 2: xe 15: 11, 12, 13 (đầy). Mức 3: bắt đầu `13 − 1 = 12` trùng, `11` trùng ⇒ `10`.

**Bài 3.** Mức 1: chèn `T` vào danh sách đã sắp. Mức 2: mô phỏng. Mức 3: trả `True` khi `T` được lên.

**Bài 4.** Mức 1: sắp hai mảng. Mức 2: mô phỏng, nhớ chỗ trống xe cuối. Mức 3: lùi.

**Bài 5.** Mức 1: tính đáp án `T`. Mức 2: mô phỏng với `T` chèn vào. Mức 3: ghi lại giờ xe khi `T` được lên.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `3`, `1`. **Bài 2:** `10`.

In [27]:
assert Solution().latestTimeCatchTheBus([3], [2, 4], 2) == 3
assert Solution().latestTimeCatchTheBus([2], [2], 1) == 1
assert Solution().latestTimeCatchTheBus([5, 15], [11, 12, 13, 14, 15], 3) == 10

def bt3(buses, passengers, capacity, T):
    return can_catch(buses, passengers, capacity, T)

def bt4(buses, passengers, capacity):
    return simulate_back(buses, passengers, capacity)

def bt5(buses, passengers, capacity):
    T = Solution().latestTimeCatchTheBus(buses, passengers, capacity)
    B = sorted(buses); P = sorted(passengers + [T]); j = 0
    for t in B:
        c = capacity
        while c and j < len(P) and P[j] <= t:
            if P[j] == T: return t
            c -= 1; j += 1
    return None

rng = random.Random(2002)
for _ in range(400):
    b = rng.sample(range(2, 40), rng.randint(1, 4)); p = rng.sample(range(2, 40), rng.randint(1, 8)); cap = rng.randint(1, 3)
    ans = brute(b, p, cap)
    assert bt3(b, p, cap, ans) and all(not bt3(b, p, cap, T) for T in range(ans + 1, max(b) + 1))
    assert bt4(b, p, cap) == ans
    assert bt5(b, p, cap) in b
print(bt5(*examples[0][:3]), bt5(*examples[1][:3]))
print("Đáp án bài tập: tất cả assert OK")

20 30
Đáp án bài tập: tất cả assert OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
buses.sort(); passengers.sort(); j = 0
for t in buses:
    seats = capacity
    while seats and j < len(passengers) and passengers[j] <= t:
        seats -= 1; j += 1
j -= 1
ans = buses[-1] if seats else passengers[j]
while j >= 0 and passengers[j] == ans:
    ans -= 1; j -= 1
return ans
```

**Checklist:**
- [ ] Sắp xếp **cả hai** mảng.
- [ ] `seats` là chỗ trống của **xe cuối**, không phải tổng.
- [ ] Đầy ⇒ bắt đầu từ người cuối **đã lên xe**; còn chỗ ⇒ từ giờ xe cuối.
- [ ] Luôn lùi qua các thời điểm trùng (cả hai trường hợp).